#  <center>Taller de Aprendizaje Automatico</center>
##  <center>Paso 3: Procesamiento de datos</center>
### <center>Procesamiento base, intento 2 para arboles</center>

Este notebook parte del mismo criterio base que `I2 Arboles`: conservar las features originales sin escalar, sin imputar y sin descartar columnas por missing, reemplazando el marcador `-999` por `NaN` para explicitar mejor los valores faltantes.

La diferencia es que en este intento 3 se agregan nuevas features derivadas, tomadas de `docs/new_features.md`, para capturar ratios fisicos, energia total del evento, distancias angulares, interacciones simples y features condicionales sobre jets. La idea es mantener el dataset apto para modelos de arboles, pero enriquecerlo con relaciones no lineales e invariantes mas informativas.

## Carga reproducible del dataset

Se cargan `training.csv` y `test.csv` desde `data/higgs-boson/`. El archivo de entrenamiento contiene `Label` y `Weight`; el archivo de test no.

In [1]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.preprocessing import OneHotEncoder
from IPython.display import display


ATTEMPT3_EPSILON = 1e-6
ATTEMPT3_SQUARED_COLUMNS = [
    "PRI_tau_pt",
    "PRI_lep_pt",
    "PRI_met",
    "DER_mass_MMC",
    "PRI_jet_leading_pt",
    "HT",
    "jets_pt_sum",
    "deltaR_tau_lep",
]
ATTEMPT3_RANK_COLUMNS = [
    "DER_mass_MMC",
    "PRI_tau_pt",
    "PRI_lep_pt",
    "PRI_met",
    "HT",
    "deltaR_tau_lep",
    "jets_pt_sum",
]


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "roadmap.md").exists() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("No se pudo localizar la raiz del proyecto.")


def replace_missing_marker_with_nan(
    data: pd.DataFrame,
    missing_marker: float = -999,
) -> pd.DataFrame:
    return data.replace(missing_marker, np.nan)


def wrap_delta_phi(delta_phi: pd.Series) -> pd.Series:
    wrapped = np.arctan2(np.sin(delta_phi), np.cos(delta_phi))
    return pd.Series(wrapped, index=delta_phi.index)


def add_attempt3_engineered_features(
    data: pd.DataFrame,
    epsilon: float = ATTEMPT3_EPSILON,
    squared_columns: list[str] | None = None,
    rank_columns: list[str] | None = None,
) -> pd.DataFrame:
    required_columns = [
        "DER_mass_MMC",
        "PRI_tau_pt",
        "PRI_tau_eta",
        "PRI_tau_phi",
        "PRI_lep_pt",
        "PRI_lep_eta",
        "PRI_lep_phi",
        "PRI_met",
        "PRI_jet_leading_pt",
        "PRI_jet_subleading_pt",
        "PRI_jet_num",
    ]
    missing_required = [column for column in required_columns if column not in data.columns]
    if missing_required:
        raise ValueError(f"Faltan columnas requeridas para generar las features del intento 3: {missing_required}")

    squared_columns = squared_columns or ATTEMPT3_SQUARED_COLUMNS
    rank_columns = rank_columns or ATTEMPT3_RANK_COLUMNS

    X = data.copy()

    leading_pt = X["PRI_jet_leading_pt"].fillna(0.0)
    subleading_pt = X["PRI_jet_subleading_pt"].fillna(0.0)
    jets_pt_sum = leading_pt + subleading_pt
    visible_energy = X["PRI_tau_pt"] + X["PRI_lep_pt"]
    ht = visible_energy + X["PRI_met"] + jets_pt_sum
    delta_eta_tau_lep = X["PRI_tau_eta"] - X["PRI_lep_eta"]
    delta_phi_tau_lep = wrap_delta_phi(X["PRI_tau_phi"] - X["PRI_lep_phi"])
    abs_phi_diff_tau_lep = delta_phi_tau_lep.abs()
    has_jet = (X["PRI_jet_num"].fillna(0) > 0).astype(np.int8)

    X["ratio_tau_lep_pt"] = X["PRI_tau_pt"] / (X["PRI_lep_pt"] + epsilon)
    X["ratio_tau_met"] = X["PRI_tau_pt"] / (X["PRI_met"] + epsilon)
    X["ratio_lep_met"] = X["PRI_lep_pt"] / (X["PRI_met"] + epsilon)
    X["ratio_mass_pt"] = X["DER_mass_MMC"] / (visible_energy + epsilon)
    X["jet_lep_ratio"] = leading_pt / (X["PRI_lep_pt"] + epsilon)
    X["HT"] = ht
    X["visible_energy"] = visible_energy
    X["met_fraction"] = X["PRI_met"] / (ht + epsilon)
    X["abs_eta_diff_tau_lep"] = delta_eta_tau_lep.abs()
    X["delta_phi_tau_lep"] = delta_phi_tau_lep
    X["abs_phi_diff_tau_lep"] = abs_phi_diff_tau_lep
    X["deltaR_tau_lep"] = np.sqrt(X["abs_eta_diff_tau_lep"] ** 2 + abs_phi_diff_tau_lep ** 2)
    X["jets_pt_sum"] = jets_pt_sum
    X["jet_lepton_balance"] = jets_pt_sum / (X["PRI_lep_pt"] + epsilon)
    X["tau_lep_interaction"] = X["PRI_tau_pt"] * X["PRI_lep_pt"]
    X["tau_met_interaction"] = X["PRI_tau_pt"] * X["PRI_met"]
    X["has_jet"] = has_jet
    X["ht_if_jets"] = ht * has_jet

    for column in squared_columns:
        if column in X.columns:
            X[f"{column}_sq"] = X[column] ** 2

    for column in rank_columns:
        if column in X.columns:
            X[f"{column}_rank"] = X[column].rank(pct=True, method="average", na_option="keep")

    return X


def get_attempt3_feature_columns(feature_columns: list[str]) -> list[str]:
    sample_data = pd.DataFrame({column: [0.0] for column in feature_columns})
    if "PRI_jet_num" in sample_data.columns:
        sample_data["PRI_jet_num"] = 0
    return add_attempt3_engineered_features(sample_data).columns.tolist()


project_root = find_project_root()
data_dir = project_root / "data" / "higgs-boson"
train_path = data_dir / "training.csv"
test_path = data_dir / "test.csv"

if not train_path.exists():
    raise FileNotFoundError(f"No se encontro el archivo esperado: {train_path}")

if not test_path.exists():
    raise FileNotFoundError(f"No se encontro el archivo esperado: {test_path}")

train_data = pd.read_csv(train_path)
test_data = pd.read_csv(test_path)

print(f"project_root = {project_root}")
print(f"train shape = {train_data.shape}")
print(f"test shape = {test_data.shape}")

project_root = /home/santig14/fing/taa/1_TallerAA
train shape = (250000, 33)
test shape = (550000, 31)


## Verificacion visual de la carga

Se muestran las primeras filas de training y test para dejar evidencia de que la carga fue correcta.

In [2]:
display(train_data.head())
display(test_data.head())


,EventId,DER_mass_MMC,DER_mass_transverse_met_lep,DER_mass_vis,DER_pt_h,DER_deltaeta_jet_jet,DER_mass_jet_jet,DER_prodeta_jet_jet,DER_deltar_tau_lep,DER_pt_tot,...,PRI_jet_num,PRI_jet_leading_pt,PRI_jet_leading_eta,PRI_jet_leading_phi,PRI_jet_subleading_pt,PRI_jet_subleading_eta,PRI_jet_subleading_phi,PRI_jet_all_pt,Weight,Label
0,100000,138.470,51.655,97.827,27.980,0.91,124.711,2.666,3.064,41.928,...,2,67.435,2.150,0.444,46.062,1.24,-2.475,113.497,0.002653,s
1,100001,160.937,68.768,103.235,48.146,-999.00,-999.000,-999.000,3.473,2.078,...,1,46.226,0.725,1.158,-999.000,-999.00,-999.000,46.226,2.233584,b
2,100002,-999.000,162.172,125.953,35.635,-999.00,-999.000,-999.000,3.148,9.336,...,1,44.251,2.053,-2.028,-999.000,-999.00,-999.000,44.251,2.347389,b
3,100003,143.905,81.417,80.943,0.414,-999.00,-999.000,-999.000,3.310,0.414,...,0,-999.000,-999.000,-999.000,-999.000,-999.00,-999.000,-0.000,5.446378,b
4,100004,175.864,16.915,134.805,16.405,-999.00,-999.000,-999.000,3.891,16.405,...,0,-999.000,-999.000,-999.000,-999.000,-999.00,-999.000,0.000,6.245333,b


,EventId,DER_mass_MMC,DER_mass_transverse_met_lep,DER_mass_vis,DER_pt_h,DER_deltaeta_jet_jet,DER_mass_jet_jet,DER_prodeta_jet_jet,DER_deltar_tau_lep,DER_pt_tot,...,PRI_met_phi,PRI_met_sumet,PRI_jet_num,PRI_jet_leading_pt,PRI_jet_leading_eta,PRI_jet_leading_phi,PRI_jet_subleading_pt,PRI_jet_subleading_eta,PRI_jet_subleading_phi,PRI_jet_all_pt
0,350000,-999.000,79.589,23.916,3.036,-999.000,-999.000,-999.000,0.903,3.036,...,2.022,98.556,0,-999.000,-999.000,-999.000,-999.000,-999.000,-999.000,-0.000
1,350001,106.398,67.490,87.949,49.994,-999.000,-999.000,-999.000,2.048,2.679,...,-1.138,176.251,1,47.575,-0.553,-0.849,-999.000,-999.000,-999.000,47.575
2,350002,117.794,56.226,96.358,4.137,-999.000,-999.000,-999.000,2.755,4.137,...,-1.868,111.505,0,-999.000,-999.000,-999.000,-999.000,-999.000,-999.000,0.000
3,350003,135.861,30.604,97.288,9.104,-999.000,-999.000,-999.000,2.811,9.104,...,1.172,164.707,0,-999.000,-999.000,-999.000,-999.000,-999.000,-999.000,0.000
4,350004,74.159,82.772,58.731,89.646,1.347,536.663,-0.339,1.028,77.213,...,-0.231,869.614,3,254.085,-1.013,-0.334,185.857,0.335,2.587,599.213


## Separacion base para entrenamiento

Antes de cualquier transformacion, se separan las columnas que no deben entrar como features del modelo:

- `EventId`: identificador del evento.
- `Weight`: peso asociado al evento, que no se usa como feature.
- `Label`: variable objetivo, que pasa a `y_train`.

Con eso se arman `X_train`, `y_train` y `X_test` como punto de partida comun.

In [3]:
excluded_train_columns = ["EventId", "Weight", "Label"]
feature_columns = [
    column for column in train_data.columns
    if column not in excluded_train_columns
]

train_event_id = train_data["EventId"].copy()
train_weight = train_data["Weight"].copy()
y_train = train_data["Label"].copy()
X_train = train_data[feature_columns].copy()

test_event_id = test_data["EventId"].copy()
X_test = test_data[feature_columns].copy()

expected_test_columns = [column for column in test_data.columns if column != "EventId"]
if feature_columns != expected_test_columns:
    raise ValueError("Las columnas de entrenamiento y test no coinciden despues de excluir EventId.")

print(f"Cantidad de features iniciales: {len(feature_columns)}")
print(f"X_train shape = {X_train.shape}")
print(f"y_train shape = {y_train.shape}")
print(f"X_test shape = {X_test.shape}")


Cantidad de features iniciales: 30
X_train shape = (250000, 30)
y_train shape = (250000,)
X_test shape = (550000, 30)


## Regla del intento 3

Este intento mantiene la filosofia de `i2_arboles`: no eliminar features por presencia de `-999`, no imputar esos valores y no reescalar ninguna columna. La limpieza minima sigue siendo reemplazar `-999` por `NaN`, pero ahora sobre esa base tambien se agregan nuevas features derivadas inspiradas en `docs/new_features.md`. Luego se mantiene el `OneHotEncoder` sobre `PRI_jet_num` para dejar el dataset listo para modelos de arboles.

In [4]:
missing_marker = -999

marker_rows = []
for column in X_train.columns:
    train_marker_count = int((X_train[column] == missing_marker).sum())
    train_marker_pct = round(train_marker_count / len(X_train) * 100, 2)
    marker_rows.append(
        {
            "feature": column,
            "train_minus_999_count": train_marker_count,
            "train_minus_999_pct": train_marker_pct,
        }
    )

marker_summary_i3 = pd.DataFrame(marker_rows).sort_values(
    ["train_minus_999_pct", "feature"],
    ascending=[False, True],
)

display(marker_summary_i3.head(10).reset_index(drop=True))
print("Columnas eliminadas en intento 3: 0")
print("En intento 3 no se descarta ninguna feature por presencia de -999.")

,feature,train_minus_999_count,train_minus_999_pct
0,DER_deltaeta_jet_jet,177457,70.98
1,DER_lep_eta_centrality,177457,70.98
2,DER_mass_jet_jet,177457,70.98
3,DER_prodeta_jet_jet,177457,70.98
4,PRI_jet_subleading_eta,177457,70.98
5,PRI_jet_subleading_phi,177457,70.98
6,PRI_jet_subleading_pt,177457,70.98
7,PRI_jet_leading_eta,99913,39.97
8,PRI_jet_leading_phi,99913,39.97
9,PRI_jet_leading_pt,99913,39.97


Columnas eliminadas en intento 3: 0
En intento 3 no se descarta ninguna feature por presencia de -999.


## Conservacion de las features originales

Como en este intento no se descarta ninguna columna, `X_train_i3` y `X_test_i3` conservan las `30` features originales del dataset. Sobre esa base primero se reemplaza `-999` por `NaN`, despues se agregan las nuevas features derivadas y finalmente se aplica el one-hot encoding de `PRI_jet_num`.

In [5]:
X_train_i3 = X_train.copy()
X_test_i3 = X_test.copy()
feature_columns_i3 = X_train_i3.columns.tolist()

print(f"X_train_i3 shape = {X_train_i3.shape}")
print(f"X_test_i3 shape = {X_test_i3.shape}")
print(f"Cantidad de features base conservadas en intento 3: {len(feature_columns_i3)}")

display(pd.DataFrame({"kept_feature": feature_columns_i3}).head(10))

X_train_i3 shape = (250000, 30)
X_test_i3 shape = (550000, 30)
Cantidad de features base conservadas en intento 3: 30


,kept_feature
0,DER_mass_MMC
1,DER_mass_transverse_met_lep
2,DER_mass_vis
3,DER_pt_h
4,DER_deltaeta_jet_jet
5,DER_mass_jet_jet
6,DER_prodeta_jet_jet
7,DER_deltar_tau_lep
8,DER_pt_tot
9,DER_sum_pt


## Reemplazo simple de `-999` por `NaN`

En este intento los valores `-999` tampoco se usan para descartar columnas ni se imputan despues. Solo se reemplazan por `NaN` para que el dataset exprese los faltantes de forma estandar antes de generar las features adicionales y de aplicar el one-hot encoding sobre `PRI_jet_num`.

In [6]:
remaining_marker_count_train = int((X_train_i3 == missing_marker).sum().sum())
remaining_marker_count_test = int((X_test_i3 == missing_marker).sum().sum())
total_cells_train_i3 = int(X_train_i3.shape[0] * X_train_i3.shape[1])
total_cells_test_i3 = int(X_test_i3.shape[0] * X_test_i3.shape[1])
remaining_marker_pct_train = remaining_marker_count_train / total_cells_train_i3 * 100
remaining_marker_pct_test = remaining_marker_count_test / total_cells_test_i3 * 100

X_train_i3_nan = replace_missing_marker_with_nan(X_train_i3, missing_marker=missing_marker)
X_test_i3_nan = replace_missing_marker_with_nan(X_test_i3, missing_marker=missing_marker)

remaining_nan_count_train = int(X_train_i3_nan.isna().sum().sum())
remaining_nan_count_test = int(X_test_i3_nan.isna().sum().sum())

print(
    f"Marcadores -999 presentes en X_train_i3 antes del reemplazo: {remaining_marker_count_train} "
    f"({remaining_marker_pct_train:.2f}% del total de celdas)"
)
print(
    f"Marcadores -999 presentes en X_test_i3 antes del reemplazo: {remaining_marker_count_test} "
    f"({remaining_marker_pct_test:.2f}% del total de celdas)"
)
print(f"NaN presentes en X_train_i3_nan despues del reemplazo: {remaining_nan_count_train}")
print(f"NaN presentes en X_test_i3_nan despues del reemplazo: {remaining_nan_count_test}")
print("No se realiza ninguna imputacion sobre estos NaN.")

Marcadores -999 presentes en X_train_i3 antes del reemplazo: 1580052 (21.07% del total de celdas)
Marcadores -999 presentes en X_test_i3 antes del reemplazo: 3473394 (21.05% del total de celdas)
NaN presentes en X_train_i3_nan despues del reemplazo: 1580052
NaN presentes en X_test_i3_nan despues del reemplazo: 3473394
No se realiza ninguna imputacion sobre estos NaN.


## Feature engineering adicional

Sobre las columnas originales ya limpiadas se agregan nuevas features inspiradas en `docs/new_features.md`. Se incluyen ratios fisicos, energia total del evento, distancias angulares, interacciones simples, indicadores condicionales por jets y algunas transformaciones no lineales sencillas. Los percentiles (`*_rank`) se calculan por separado en train y test para evitar mezclar ambas tablas en esta etapa.

In [7]:
X_train_i3_engineered = add_attempt3_engineered_features(X_train_i3_nan)
X_test_i3_engineered = add_attempt3_engineered_features(X_test_i3_nan)

feature_columns_i3_engineered = X_train_i3_engineered.columns.tolist()
engineered_feature_columns_i3 = [
    column for column in feature_columns_i3_engineered
    if column not in feature_columns_i3
]

if feature_columns_i3_engineered != X_test_i3_engineered.columns.tolist():
    raise ValueError("Train y test no quedaron con las mismas columnas despues del feature engineering del intento 3.")

engineered_nan_count_train = int(X_train_i3_engineered[engineered_feature_columns_i3].isna().sum().sum())
engineered_nan_count_test = int(X_test_i3_engineered[engineered_feature_columns_i3].isna().sum().sum())

print(f"X_train_i3_engineered shape = {X_train_i3_engineered.shape}")
print(f"X_test_i3_engineered shape = {X_test_i3_engineered.shape}")
print(f"Cantidad de nuevas features agregadas: {len(engineered_feature_columns_i3)}")
print(f"NaN presentes en las nuevas features de train: {engineered_nan_count_train}")
print(f"NaN presentes en las nuevas features de test: {engineered_nan_count_test}")

display(pd.DataFrame({"engineered_feature": engineered_feature_columns_i3}))

X_train_i3_engineered shape = (250000, 63)
X_test_i3_engineered shape = (550000, 63)
Cantidad de nuevas features agregadas: 33
NaN presentes en las nuevas features de train: 214255
NaN presentes en las nuevas features de test: 471622


,engineered_feature
0,ratio_tau_lep_pt
1,ratio_tau_met
2,ratio_lep_met
3,ratio_mass_pt
4,jet_lep_ratio
5,HT
6,visible_energy
7,met_fraction
8,abs_eta_diff_tau_lep
9,delta_phi_tau_lep


## One-hot encoding de `PRI_jet_num`

La columna `PRI_jet_num` sigue representando una cantidad discreta de jets. Despues de agregar las nuevas features derivadas, se vuelve a tratar esta columna como categorica y se transforma con `OneHotEncoder`, ajustando solo con training y replicando luego sobre test. El resto de las columnas numericas, incluidas las derivadas, se conserva sin escalar y con los faltantes expresados como `NaN`.

In [8]:
jet_num_column = "PRI_jet_num"
remaining_numeric_columns_i3 = [column for column in feature_columns_i3_engineered if column != jet_num_column]

jet_num_encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

X_train_i3_jet_num = X_train_i3_engineered[[jet_num_column]].astype(int)
X_test_i3_jet_num = X_test_i3_engineered[[jet_num_column]].astype(int)

X_train_i3_jet_num_encoded_array = jet_num_encoder.fit_transform(X_train_i3_jet_num)
jet_num_encoded_columns = jet_num_encoder.get_feature_names_out([jet_num_column]).tolist()

X_train_i3_jet_num_encoded = pd.DataFrame(
    X_train_i3_jet_num_encoded_array,
    columns=jet_num_encoded_columns,
    index=X_train_i3_engineered.index,
)

X_test_i3_jet_num_encoded = pd.DataFrame(
    jet_num_encoder.transform(X_test_i3_jet_num),
    columns=jet_num_encoded_columns,
    index=X_test_i3_engineered.index,
)

print(f"Columnas generadas para {jet_num_column}: {jet_num_encoded_columns}")
print(f"X_train_i3_jet_num_encoded shape = {X_train_i3_jet_num_encoded.shape}")
print(f"X_test_i3_jet_num_encoded shape = {X_test_i3_jet_num_encoded.shape}")

Columnas generadas para PRI_jet_num: ['PRI_jet_num_0', 'PRI_jet_num_1', 'PRI_jet_num_2', 'PRI_jet_num_3']
X_train_i3_jet_num_encoded shape = (250000, 4)
X_test_i3_jet_num_encoded shape = (550000, 4)


## Armado final sin escalado ni imputacion

Despues de codificar `PRI_jet_num`, el resto de las columnas numericas se conserva en su escala original. Eso incluye tanto las features base como las nuevas features derivadas. Los faltantes siguen expresados como `NaN` y el dataset final se arma concatenando esas columnas sin escalar con las nuevas columnas binarias del one-hot encoding.

In [9]:
X_train_i3_passthrough_numeric = X_train_i3_engineered[remaining_numeric_columns_i3].copy()
X_test_i3_passthrough_numeric = X_test_i3_engineered[remaining_numeric_columns_i3].copy()

X_train_i3_final = pd.concat([X_train_i3_passthrough_numeric, X_train_i3_jet_num_encoded], axis=1)
X_test_i3_final = pd.concat([X_test_i3_passthrough_numeric, X_test_i3_jet_num_encoded], axis=1)
feature_columns_i3_final = X_train_i3_final.columns.tolist()

retained_nan_count_train = int(X_train_i3_final[remaining_numeric_columns_i3].isna().sum().sum())
retained_nan_count_test = int(X_test_i3_final[remaining_numeric_columns_i3].isna().sum().sum())
retained_marker_count_train = int((X_train_i3_final[remaining_numeric_columns_i3] == missing_marker).sum().sum())
retained_marker_count_test = int((X_test_i3_final[remaining_numeric_columns_i3] == missing_marker).sum().sum())

print(f"X_train_i3_final shape = {X_train_i3_final.shape}")
print(f"X_test_i3_final shape = {X_test_i3_final.shape}")
print(f"Cantidad final de features despues de one-hot encoding: {len(feature_columns_i3_final)}")
print(f"NaN conservados en X_train_i3_final: {retained_nan_count_train}")
print(f"NaN conservados en X_test_i3_final: {retained_nan_count_test}")
print(f"Marcadores -999 restantes en X_train_i3_final: {retained_marker_count_train}")
print(f"Marcadores -999 restantes en X_test_i3_final: {retained_marker_count_test}")

display(pd.DataFrame({"final_feature": feature_columns_i3_final}).head(20))

X_train_i3_final shape = (250000, 66)
X_test_i3_final shape = (550000, 66)
Cantidad final de features despues de one-hot encoding: 66
NaN conservados en X_train_i3_final: 1794307
NaN conservados en X_test_i3_final: 3945016
Marcadores -999 restantes en X_train_i3_final: 0
Marcadores -999 restantes en X_test_i3_final: 0


,final_feature
0,DER_mass_MMC
1,DER_mass_transverse_met_lep
2,DER_mass_vis
3,DER_pt_h
4,DER_deltaeta_jet_jet
5,DER_mass_jet_jet
6,DER_prodeta_jet_jet
7,DER_deltar_tau_lep
8,DER_pt_tot
9,DER_sum_pt


## Objetos resultantes del intento 3

Se muestran las primeras filas de `X_train_i3_final` y de `y_train` para confirmar que la separacion sigue bien y que el intento 3 queda listo sin descarte de columnas, sin imputacion y sin escalado, pero con las nuevas features derivadas ya incorporadas y los faltantes expresados como `NaN`.

In [10]:
display(X_train_i3_final.head())
display(y_train.head().to_frame(name="Label"))

,DER_mass_MMC,DER_mass_transverse_met_lep,DER_mass_vis,DER_pt_h,DER_deltaeta_jet_jet,DER_mass_jet_jet,DER_prodeta_jet_jet,DER_deltar_tau_lep,DER_pt_tot,DER_sum_pt,...,PRI_tau_pt_rank,PRI_lep_pt_rank,PRI_met_rank,HT_rank,deltaR_tau_lep_rank,jets_pt_sum_rank,PRI_jet_num_0,PRI_jet_num_1,PRI_jet_num_2,PRI_jet_num_3
0,138.470,51.655,97.827,27.980,0.91,124.711,2.666,3.064,41.928,197.760,...,0.521738,0.724802,0.169736,0.690796,0.816564,0.773588,0.0,0.0,1.0,0.0
1,160.937,68.768,103.235,48.146,NaN,NaN,NaN,3.473,2.078,125.157,...,0.707304,0.399814,0.660102,0.555176,0.953792,0.535916,0.0,1.0,0.0,0.0
2,NaN,162.172,125.953,35.635,NaN,NaN,NaN,3.148,9.336,197.814,...,0.509350,0.986128,0.774778,0.774840,0.874164,0.524668,0.0,1.0,0.0,0.0
3,143.905,81.417,80.943,0.414,NaN,NaN,NaN,3.310,0.414,75.968,...,0.155246,0.749066,0.432400,0.228420,0.928276,0.199828,1.0,0.0,0.0,0.0
4,175.864,16.915,134.805,16.405,NaN,NaN,NaN,3.891,16.405,57.983,...,0.390132,0.152328,0.005542,0.004754,0.981984,0.199828,1.0,0.0,0.0,0.0


,Label
0,s
1,b
2,b
3,b
4,b


## Pipeline final reutilizable

A continuacion se deja una version reusable del flujo completo del intento 3. Esta version parte desde la carga de datos, separa columnas especiales, conserva todas las features, reemplaza `-999` por `NaN`, genera las nuevas features derivadas de `docs/new_features.md`, aplica `OneHotEncoder` sobre `PRI_jet_num` y devuelve los datasets transformados listos para entrenar modelos de arboles.

In [11]:
def load_higgs_data(project_root: Path | None = None) -> tuple[pd.DataFrame, pd.DataFrame]:
    root = find_project_root(project_root)
    data_dir = root / "data" / "higgs-boson"
    train_path = data_dir / "training.csv"
    test_path = data_dir / "test.csv"

    if not train_path.exists():
        raise FileNotFoundError(f"No se encontro el archivo esperado: {train_path}")

    if not test_path.exists():
        raise FileNotFoundError(f"No se encontro el archivo esperado: {test_path}")

    return pd.read_csv(train_path), pd.read_csv(test_path)


def split_higgs_datasets(
    train_data: pd.DataFrame,
    test_data: pd.DataFrame,
) -> tuple[pd.Series, pd.Series, pd.Series, pd.Series, pd.DataFrame, pd.DataFrame]:
    excluded_train_columns = ["EventId", "Weight", "Label"]
    feature_columns = [
        column for column in train_data.columns
        if column not in excluded_train_columns
    ]

    expected_test_columns = [column for column in test_data.columns if column != "EventId"]
    if feature_columns != expected_test_columns:
        raise ValueError("Las columnas de entrenamiento y test no coinciden despues de excluir EventId.")

    train_event_id = train_data["EventId"].copy()
    train_weight = train_data["Weight"].copy()
    y_train = train_data["Label"].copy()
    X_train = train_data[feature_columns].copy()

    test_event_id = test_data["EventId"].copy()
    X_test = test_data[feature_columns].copy()

    return train_event_id, train_weight, y_train, test_event_id, X_train, X_test


def summarize_attempt3_marker_usage(
    X_train: pd.DataFrame,
    missing_marker: float = -999,
) -> pd.DataFrame:
    marker_summary = pd.DataFrame(
        {
            "feature": X_train.columns,
            "train_minus_999_count": [(X_train[column] == missing_marker).sum() for column in X_train.columns],
        }
    )
    marker_summary["train_minus_999_pct"] = (
        marker_summary["train_minus_999_count"] / len(X_train) * 100
    ).round(2)
    marker_summary = marker_summary.sort_values(
        ["train_minus_999_pct", "feature"],
        ascending=[False, True],
    )

    return marker_summary


def build_attempt3_preprocessing_pipeline(
    feature_columns: list[str],
    jet_num_column: str = "PRI_jet_num",
    missing_marker: float = -999,
) -> tuple[Pipeline, list[str], list[str], list[str]]:
    if jet_num_column not in feature_columns:
        raise ValueError(f"La columna {jet_num_column} debe existir en feature_columns.")

    all_feature_columns_i3 = get_attempt3_feature_columns(feature_columns)
    numeric_columns_i3 = [column for column in all_feature_columns_i3 if column != jet_num_column]
    engineered_feature_columns_i3 = [
        column for column in all_feature_columns_i3
        if column not in feature_columns
    ]

    replace_missing = FunctionTransformer(
        replace_missing_marker_with_nan,
        validate=False,
        feature_names_out="one-to-one",
        kw_args={"missing_marker": missing_marker},
    )
    feature_engineering = FunctionTransformer(
        add_attempt3_engineered_features,
        validate=False,
        feature_names_out=lambda transformer, input_features: get_attempt3_feature_columns(list(input_features)),
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("numeric", "passthrough", numeric_columns_i3),
            ("jet_num", OneHotEncoder(sparse_output=False, handle_unknown="ignore"), [jet_num_column]),
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    )

    preprocessing_pipeline = Pipeline(
        steps=[
            ("replace_missing_marker", replace_missing),
            ("feature_engineering", feature_engineering),
            ("preprocessor", preprocessor),
        ]
    )

    return preprocessing_pipeline, numeric_columns_i3, [jet_num_column], engineered_feature_columns_i3


def transform_attempt3_with_pipeline(
    train_data: pd.DataFrame,
    test_data: pd.DataFrame,
    missing_marker: float = -999,
) -> dict[str, object]:
    train_event_id, train_weight, y_train, test_event_id, X_train, X_test = split_higgs_datasets(
        train_data,
        test_data,
    )

    feature_columns_i3 = X_train.columns.tolist()
    drop_columns_i3: list[str] = []
    marker_summary_i3 = summarize_attempt3_marker_usage(
        X_train,
        missing_marker=missing_marker,
    )

    (
        preprocessing_pipeline,
        numeric_columns_i3,
        categorical_columns_i3,
        engineered_feature_columns_i3,
    ) = build_attempt3_preprocessing_pipeline(
        feature_columns_i3,
        missing_marker=missing_marker,
    )

    X_train_selected = X_train[feature_columns_i3].copy()
    X_test_selected = X_test[feature_columns_i3].copy()

    X_train_processed_array = preprocessing_pipeline.fit_transform(X_train_selected)
    X_test_processed_array = preprocessing_pipeline.transform(X_test_selected)

    processed_feature_names = preprocessing_pipeline.named_steps["preprocessor"].get_feature_names_out().tolist()

    X_train_processed = pd.DataFrame(
        X_train_processed_array,
        columns=processed_feature_names,
        index=X_train_selected.index,
    )

    X_test_processed = pd.DataFrame(
        X_test_processed_array,
        columns=processed_feature_names,
        index=X_test_selected.index,
    )

    return {
        "train_event_id": train_event_id,
        "train_weight": train_weight,
        "y_train": y_train,
        "test_event_id": test_event_id,
        "feature_columns_i3": feature_columns_i3,
        "engineered_feature_columns_i3": engineered_feature_columns_i3,
        "drop_columns_i3": drop_columns_i3,
        "marker_summary_i3": marker_summary_i3,
        "numeric_columns_i3": numeric_columns_i3,
        "categorical_columns_i3": categorical_columns_i3,
        "processed_feature_names": processed_feature_names,
        "preprocessing_pipeline": preprocessing_pipeline,
        "X_train_processed": X_train_processed,
        "X_test_processed": X_test_processed,
    }


train_data_pipeline, test_data_pipeline = load_higgs_data()
attempt3_pipeline_artifacts = transform_attempt3_with_pipeline(train_data_pipeline, test_data_pipeline)

print(f"Columnas descartadas en intento 3: {attempt3_pipeline_artifacts['drop_columns_i3']}")
print(f"Cantidad de features base conservadas antes del pipeline: {len(attempt3_pipeline_artifacts['feature_columns_i3'])}")
print(f"Cantidad de nuevas features derivadas: {len(attempt3_pipeline_artifacts['engineered_feature_columns_i3'])}")
print(f"Cantidad final de features procesadas: {len(attempt3_pipeline_artifacts['processed_feature_names'])}")
print(f"X_train_processed shape = {attempt3_pipeline_artifacts['X_train_processed'].shape}")
print(f"X_test_processed shape = {attempt3_pipeline_artifacts['X_test_processed'].shape}")
print(
    f"NaN en X_train_processed: {int(attempt3_pipeline_artifacts['X_train_processed'].isna().sum().sum())}"
)
print(
    f"NaN en X_test_processed: {int(attempt3_pipeline_artifacts['X_test_processed'].isna().sum().sum())}"
)

display(attempt3_pipeline_artifacts['X_train_processed'].head())
display(attempt3_pipeline_artifacts['y_train'].head().to_frame(name='Label'))

Columnas descartadas en intento 3: []
Cantidad de features base conservadas antes del pipeline: 30
Cantidad de nuevas features derivadas: 33
Cantidad final de features procesadas: 66
X_train_processed shape = (250000, 66)
X_test_processed shape = (550000, 66)
NaN en X_train_processed: 1794307
NaN en X_test_processed: 3945016


,DER_mass_MMC,DER_mass_transverse_met_lep,DER_mass_vis,DER_pt_h,DER_deltaeta_jet_jet,DER_mass_jet_jet,DER_prodeta_jet_jet,DER_deltar_tau_lep,DER_pt_tot,DER_sum_pt,...,PRI_tau_pt_rank,PRI_lep_pt_rank,PRI_met_rank,HT_rank,deltaR_tau_lep_rank,jets_pt_sum_rank,PRI_jet_num_0,PRI_jet_num_1,PRI_jet_num_2,PRI_jet_num_3
0,138.470,51.655,97.827,27.980,0.91,124.711,2.666,3.064,41.928,197.760,...,0.521738,0.724802,0.169736,0.690796,0.816564,0.773588,0.0,0.0,1.0,0.0
1,160.937,68.768,103.235,48.146,NaN,NaN,NaN,3.473,2.078,125.157,...,0.707304,0.399814,0.660102,0.555176,0.953792,0.535916,0.0,1.0,0.0,0.0
2,NaN,162.172,125.953,35.635,NaN,NaN,NaN,3.148,9.336,197.814,...,0.509350,0.986128,0.774778,0.774840,0.874164,0.524668,0.0,1.0,0.0,0.0
3,143.905,81.417,80.943,0.414,NaN,NaN,NaN,3.310,0.414,75.968,...,0.155246,0.749066,0.432400,0.228420,0.928276,0.199828,1.0,0.0,0.0,0.0
4,175.864,16.915,134.805,16.405,NaN,NaN,NaN,3.891,16.405,57.983,...,0.390132,0.152328,0.005542,0.004754,0.981984,0.199828,1.0,0.0,0.0,0.0


,Label
0,s
1,b
2,b
3,b
4,b


## Guardado de los datos procesados

Para cerrar el intento 3, se guardan los datasets procesados y los artefactos principales en `data/higgs-boson/processed/i3_arboles/`. Esto deja disponible una salida reutilizable para los siguientes pasos de modelado sin tener que reejecutar manualmente todas las transformaciones.

In [12]:
def save_attempt3_processed_data(
    artifacts: dict[str, object],
    output_dir: Path,
) -> list[Path]:
    output_dir.mkdir(parents=True, exist_ok=True)

    files_to_save: dict[str, pd.DataFrame] = {
        "X_train_processed.csv": artifacts["X_train_processed"],
        "X_test_processed.csv": artifacts["X_test_processed"],
        "y_train.csv": artifacts["y_train"].to_frame(name="Label"),
        "train_weight.csv": artifacts["train_weight"].to_frame(name="Weight"),
        "train_event_id.csv": artifacts["train_event_id"].to_frame(name="EventId"),
        "test_event_id.csv": artifacts["test_event_id"].to_frame(name="EventId"),
        "drop_columns_i3.csv": pd.DataFrame({"dropped_feature": artifacts["drop_columns_i3"]}),
        "feature_columns_i3.csv": pd.DataFrame({"feature": artifacts["feature_columns_i3"]}),
        "engineered_feature_columns_i3.csv": pd.DataFrame({"feature": artifacts["engineered_feature_columns_i3"]}),
        "processed_feature_names.csv": pd.DataFrame({"feature": artifacts["processed_feature_names"]}),
        "marker_summary_i3.csv": artifacts["marker_summary_i3"],
    }

    saved_paths: list[Path] = []
    for filename, dataframe in files_to_save.items():
        file_path = output_dir / filename
        dataframe.to_csv(file_path, index=False)
        saved_paths.append(file_path)

    return saved_paths


processed_output_dir = find_project_root() / "data" / "higgs-boson" / "processed" / "i3_arboles"
saved_paths_i3 = save_attempt3_processed_data(attempt3_pipeline_artifacts, processed_output_dir)

print(f"Directorio de salida: {processed_output_dir}")
print(f"Cantidad de archivos guardados: {len(saved_paths_i3)}")
for path in saved_paths_i3:
    print(f"- {path}")

Directorio de salida: /home/santig14/fing/taa/1_TallerAA/data/higgs-boson/processed/i3_arboles
Cantidad de archivos guardados: 11
- /home/santig14/fing/taa/1_TallerAA/data/higgs-boson/processed/i3_arboles/X_train_processed.csv
- /home/santig14/fing/taa/1_TallerAA/data/higgs-boson/processed/i3_arboles/X_test_processed.csv
- /home/santig14/fing/taa/1_TallerAA/data/higgs-boson/processed/i3_arboles/y_train.csv
- /home/santig14/fing/taa/1_TallerAA/data/higgs-boson/processed/i3_arboles/train_weight.csv
- /home/santig14/fing/taa/1_TallerAA/data/higgs-boson/processed/i3_arboles/train_event_id.csv
- /home/santig14/fing/taa/1_TallerAA/data/higgs-boson/processed/i3_arboles/test_event_id.csv
- /home/santig14/fing/taa/1_TallerAA/data/higgs-boson/processed/i3_arboles/drop_columns_i3.csv
- /home/santig14/fing/taa/1_TallerAA/data/higgs-boson/processed/i3_arboles/feature_columns_i3.csv
- /home/santig14/fing/taa/1_TallerAA/data/higgs-boson/processed/i3_arboles/engineered_feature_columns_i3.csv
- /home/s

## Tabla de correlacion

Al final del intento 3 se calcula la matriz de correlacion de Pearson sobre `X_train_i3_final`, ya con las features derivadas incorporadas. Tambien se agrega una tabla ordenada por correlacion absoluta contra el target binario (`s=1`, `b=0`) para identificar rapido las columnas mas asociadas a la clase.

In [13]:
correlation_table_i3 = X_train_i3_final.corr(numeric_only=True)
y_train_i3_binary = y_train.map({"b": 0, "s": 1}).astype(np.int8)
correlation_with_target_i3 = (
    X_train_i3_final.assign(target_signal=y_train_i3_binary)
    .corr(numeric_only=True)[["target_signal"]]
    .drop(index="target_signal")
    .rename(columns={"target_signal": "corr_with_target_signal"})
)
correlation_with_target_i3["abs_corr_with_target_signal"] = correlation_with_target_i3["corr_with_target_signal"].abs()
correlation_with_target_i3 = correlation_with_target_i3.sort_values(
    "abs_corr_with_target_signal",
    ascending=False,
)

print(f"Matriz de correlacion shape = {correlation_table_i3.shape}")
display(correlation_table_i3.round(3))
print("Top 20 features por correlacion absoluta con el target:")
display(correlation_with_target_i3.head(20).round(3))

Matriz de correlacion shape = (66, 66)


,DER_mass_MMC,DER_mass_transverse_met_lep,DER_mass_vis,DER_pt_h,DER_deltaeta_jet_jet,DER_mass_jet_jet,DER_prodeta_jet_jet,DER_deltar_tau_lep,DER_pt_tot,DER_sum_pt,...,PRI_tau_pt_rank,PRI_lep_pt_rank,PRI_met_rank,HT_rank,deltaR_tau_lep_rank,jets_pt_sum_rank,PRI_jet_num_0,PRI_jet_num_1,PRI_jet_num_2,PRI_jet_num_3
DER_mass_MMC,1.000,0.221,0.912,0.024,-0.027,-0.022,0.019,0.497,0.014,0.125,...,0.251,0.307,0.118,0.189,0.446,0.018,-0.015,0.004,0.007,0.008
DER_mass_transverse_met_lep,0.221,1.000,0.190,-0.249,-0.134,-0.132,0.124,0.043,0.018,-0.147,...,-0.202,0.317,0.312,-0.036,0.037,-0.256,0.220,-0.060,-0.155,-0.063
DER_mass_vis,0.912,0.190,1.000,-0.063,-0.039,-0.055,0.033,0.580,-0.001,0.089,...,0.274,0.369,-0.081,0.116,0.525,-0.038,0.014,0.017,-0.024,-0.017
DER_pt_h,0.024,-0.249,-0.063,1.000,-0.038,0.189,-0.026,-0.539,0.311,0.833,...,0.273,0.233,0.485,0.744,-0.519,0.765,-0.565,0.084,0.356,0.334
DER_deltaeta_jet_jet,-0.027,-0.134,-0.039,-0.038,1.000,0.797,-0.836,0.031,-0.148,-0.104,...,0.057,-0.068,-0.053,-0.057,0.018,-0.036,NaN,NaN,0.175,-0.175
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
jets_pt_sum_rank,0.018,-0.256,-0.038,0.765,-0.036,0.291,-0.050,-0.435,0.276,0.820,...,0.223,0.172,0.374,0.909,-0.448,1.000,-0.877,0.191,0.559,0.411
PRI_jet_num_0,-0.015,0.220,0.014,-0.565,NaN,NaN,NaN,0.304,-0.187,-0.579,...,-0.159,-0.113,-0.244,-0.773,0.330,-0.877,1.000,-0.547,-0.410,-0.254
PRI_jet_num_1,0.004,-0.060,0.017,0.084,NaN,NaN,NaN,-0.029,-0.068,-0.047,...,0.013,0.006,-0.007,0.151,-0.050,0.191,-0.547,1.000,-0.337,-0.209
PRI_jet_num_2,0.007,-0.155,-0.024,0.356,0.175,0.074,-0.124,-0.201,-0.037,0.379,...,0.123,0.068,0.175,0.501,-0.211,0.559,-0.410,-0.337,1.000,-0.157


Top 20 features por correlacion absoluta con el target:


,corr_with_target_signal,abs_corr_with_target_signal
DER_mass_transverse_met_lep,-0.351,0.351
DER_deltaeta_jet_jet,0.328,0.328
DER_mass_jet_jet,0.317,0.317
PRI_tau_pt_rank,0.316,0.316
DER_lep_eta_centrality,0.308,0.308
DER_prodeta_jet_jet,-0.294,0.294
DER_met_phi_centrality,0.272,0.272
ratio_tau_lep_pt,0.249,0.249
met_fraction,-0.249,0.249
PRI_tau_pt,0.235,0.235
